Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo
from application.peak_processor import Peak_Processor

%run "plot_style_kalinka.py"


In [ ]:
peak_processor = Peak_Processor()

### Estimate Coincidence Window

In [ ]:
DetectorHeight = 100 #mm
DetectorDiameter = 100 #mm
DetectorLongestLength = (DetectorHeight**2 + DetectorDiameter**2)**0.5

SpeedofLight = 299792458 # m/s
CoincidenceTime = DetectorLongestLength / SpeedofLight * 1e9 # in ns

In [ ]:
CoincidenceTime

### Area Spectrum

#### 3-fold 5 PE (density hist; not scaled)

In [ ]:

dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/Co57_voltage_47_peak_info_1.csv",
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/Cs137_voltage_47_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/gain_calibration_voltage_47_peak_info_12.csv",

}

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        plt.hist(d2d_data.sum_area_PE, 
                alpha=0.5, label=f'{key}',
                range=[-0.1,5000],
                bins=200,
                density=True, 
        )

plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
plt.yscale('log')
plt.legend()

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/summed_area_spectrum_full.pdf", dpi=100, bbox_inches='tight')

#### 22-fold coincidence 3 PE threshold

In [ ]:

dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co22_3PE/Co57_voltage_all_peak_info.csv",
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co22_3PE/Cs137_voltage_all_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co22_3PE/gain_calibration_voltage_47_peak_info_12.csv",
    "tritium": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co22_3PE/tritium_voltage_46_peak_info.csv"

}

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        plt.hist(d2d_data.sum_area_PE, 
                alpha=0.5, label=f'{key}',
                range=[-0.1,5000],
                bins=200,
                density=True, 
        )

plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
plt.yscale('log')
plt.legend()

# plt.savefig("/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/summed_area_spectrum_full.pdf", dpi=100, bbox_inches='tight')

#### 13-fold coincidence, 5PE threshold

In [ ]:
dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/Co57_voltage_all_peak_info.csv",
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/Cs137_voltage_all_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/gain_calibration_voltage_47_peak_info_12.csv",
    "tritium": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/tritium_voltage_46_peak_info.csv"
}


for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        plt.hist(d2d_data.sum_area_PE, bins=100,
        alpha=0.7, label=f'{key}',
        range=[-0.1,5000],
        density=True, 
        )

plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
# plt.yscale('log')
plt.legend()

#### 13-fold coincidence, 5PE threshold, scale to match background rising edge

In [ ]:
dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/Co57_voltage_all_peak_info.csv",
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/Cs137_voltage_all_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/gain_calibration_voltage_47_peak_info_12.csv",
    "tritium": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co13_5PE_2/tritium_voltage_46_peak_info.csv"
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[0,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)

# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        plt.bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )


plt.xlabel('Sum Area [PE]')
plt.ylabel('Counts')

         
#log y
# plt.yscale('log')
plt.legend()

#### Individual spectrum (scaled)

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

dict_source_path = {
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/Cs137_voltage_47_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/gain_calibration_voltage_47_peak_info_12.csv",
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[50,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[0:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)


# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        ax[0].bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )

ax[1].scatter(bin_edges[:-1], hist_list[0]-hist_list[1],
                )
# ax[1].set_ylim(0,None)
# log y

plt.xlabel('Summed area [PE]')
ax[0].set_ylabel('Scaled density')
ax[1].set_ylabel('Diff')

         
#log y
# plt.yscale('log')
ax[0].legend()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/Co57_voltage_47_peak_info_1.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_5PE/gain_calibration_voltage_47_peak_info_12.csv",
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[50,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[0:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)


# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        ax[0].bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )

ax[1].scatter(bin_edges[:-1], hist_list[0]-hist_list[1],
                )
# ax[1].set_ylim(0,None)
# log y

plt.xlabel('Summed area [PE]')
ax[0].set_ylabel('Scaled density')
ax[1].set_ylabel('Diff')

         
#log y
# plt.yscale('log')
ax[0].legend()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

dict_source_path = {
    "Co57": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/Co57_voltage_47_sum_area_PE_co3.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/gain_calibration_voltage_47_sum_area_PE_co3.csv",
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[50,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[0:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)


# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        ax[0].bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )

ax[1].scatter(bin_edges[:-1], hist_list[0]-hist_list[1],
                )
# ax[1].set_ylim(0,None)
# log y

plt.xlabel('Summed area [PE]')
ax[0].set_ylabel('Scaled density')
ax[1].set_ylabel('Diff')

         
#log y
# plt.yscale('log')
ax[0].legend()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

dict_source_path = {
    "Cs137": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/Cs137_voltage_47_sum_area_PE_co3.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/gain_calibration_voltage_47_sum_area_PE_co3.csv",
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[50,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[0:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)


# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        ax[0].bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )

ax[1].scatter(bin_edges[:-1], hist_list[0]-hist_list[1],
                )
# ax[1].set_ylim(0,None)
# log y

plt.xlabel('Summed area [PE]')
ax[0].set_ylabel('Scaled density')
ax[1].set_ylabel('Diff')

         
#log y
# plt.yscale('log')
ax[0].legend()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(10, 6), sharex=True, height_ratios=[3, 1])
fig.subplots_adjust(hspace=0)

dict_source_path = {
    "tritium": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/tritium_voltage_46_peak_info.csv",
    "background": "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spectrum/co3_3PE/gain_calibration_voltage_47_sum_area_PE_co3.csv",
}

hist_list = []
bin_edges_list = []
max_height = 0

for key, path in dict_source_path.items():

        df = pd.read_csv(path,
                delimiter=",",
                quotechar='"', 
                skipinitialspace=True, 
                encoding="utf-8")

        d2d_data = d2d.data(df)

        hist, bin_edges = np.histogram(d2d_data.sum_area_PE, bins=100,
                range=[50,5000],
                density=True, 
        )
        

        tmp = hist[:10].max()
        if tmp > max_height: 
                max_height = tmp
                max_bin_id = hist[:10].argmax()

        hist_list.append(hist)
        bin_edges_list.append(bin_edges)


# scale histogram to have the same height in the first 10 bins
for i, (hist, bin_edges) in enumerate(zip(hist_list, bin_edges_list)):
        # hist_list[i] = hist / hist[:10].max() * max_height
        ratio = max_height / hist[max_bin_id]
        hist = hist * ratio

        ax[0].bar(bin_edges[:-1], hist,
                label=f'{list(dict_source_path.keys())[i]}',
                width=50, 
                alpha = 0.5
                )

ax[1].scatter(bin_edges[:-1], hist_list[0]-hist_list[1],
                )
# ax[1].set_ylim(0,None)
# log y

plt.xlabel('Summed area [PE]')
ax[0].set_ylabel('Scaled density')
ax[1].set_ylabel('Diff')

         
#log y
# plt.yscale('log')
ax[0].legend()